In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [17]:
df = pd.read_csv('Task 3 and 4_Loan_Data.csv', index_col = 0)

In [18]:
x = df.drop(columns = ["default"])
y = df["default"]
#here I am taking a random (but set state so it is the same when rerun) selection of the data to train the model on and leaving 30% to test
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify=y
)

First, I will train a logistic regression model on the data, scaling the features as well 

In [19]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(x_train_scaled, y_train)
y_pred_lr = log_reg.predict_proba(x_test_scaled)[:,1]

And now a decision tree using random forest classifier from scikit-learn

In [20]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(x_train, y_train)
y_pred_rf = rf.predict_proba(x_test)[:,1]

In [21]:
print("LR ROC-AUC:", roc_auc_score(y_test, y_pred_lr))
print("RF ROC-AUC:", roc_auc_score(y_test, y_pred_rf))

LR ROC-AUC: 0.9999896829344681
RF ROC-AUC: 0.9998533502827982


The logistic regression model is better, as the ROC-AUC is higher (closer to 1) so we will use this model.

In [22]:
def expected_loss(borrower, model, scaler, recovery_rate = 0.1):
    x_new = pd.DataFrame([borrower])
    if scaler:
        x_new = scaler.transform(x_new)
    
    prob = model.predict_proba(x_new)[:,1][0]

    loss_rate = 1 - recovery_rate

    loan_amount = borrower["loan_amt_outstanding"]

    expected_l = prob * loss_rate * loan_amount

    return expected_l, prob



In [27]:
borrower = {
    "credit_lines_outstanding": 3,
    "loan_amt_outstanding": 10000,
    "total_debt_outstanding": 13000,
    "income": 20000,
    "years_employed": 5,
    "fico_score": 700
}

el_lr, pd_lr = expected_loss(borrower, log_reg, scaler)
print("Probability", pd_lr, "Expected Loss:", el_lr)

Probability 0.7659987209064485 Expected Loss: 6893.988488158037
